# Personalization Engine

This notebook builds personalized recommendations by learning from simulated user feedback generated in the previous notebook.

Unlike the Recommendation Engine, which predicts relevant products, the Personalization Engine adapts recommendations to each user's historical preferences and interactions.

The final output is a personalized recommendation list and a user preference profile for every customer.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## Load Feedback-Adjusted Recommendations

The input for this notebook is the output generated by the Feedback System.

This dataset already contains recommendation scores adjusted using simulated user feedback.

In [ ]:
recommendations_df = pd.read_csv(
    "../data/features/personalized_recommendations.csv"
)

recommendations_df.head()

In [ ]:
recommendations_df.info()

In [ ]:
recommendations_df.describe()

In [ ]:
recommendations_df.isnull().sum()

## Understanding Personalization

Recommendation systems identify products that are likely to interest users.

Personalization systems go one step further by adapting recommendations according to each user's behavior and feedback history.

In this notebook, user preference profiles are created using recommendation scores and simulated feedback.

## Build User Preference Profiles

Each user's historical recommendations and feedback are summarized into a preference profile.

These profiles capture how users typically respond to recommended products and serve as the foundation for personalized ranking.

In [ ]:
user_profiles = (
    recommendations_df
    .groupby("visitorid")
    .agg(
        avg_recommendation_score=("recommendation_score", "mean"),
        avg_updated_score=("updated_score", "mean"),
        avg_feedback_score=("feedback_score", "mean"),
        total_recommendations=("itemid", "count"),
        positive_feedback=("feedback", lambda x: (x == "Like").sum()),
        neutral_feedback=("feedback", lambda x: (x == "Neutral").sum()),
        negative_feedback=("feedback", lambda x: (x == "Dislike").sum()),
        max_updated_score=("updated_score", "max"),
        min_updated_score=("updated_score", "min"),
    )
    .reset_index()
)

user_profiles.head()

In [ ]:
user_profiles.describe()

In [ ]:
user_profiles.head(10)

## Merge User Profiles

The user preference statistics are merged back into the recommendation dataset.

This allows personalization to consider both product-level and user-level information simultaneously.

In [ ]:
recommendations_df = recommendations_df.merge(
    user_profiles,
    on="visitorid",
    how="left"
)

recommendations_df.head()

## Calculate Personalization Score

The personalization score combines recommendation confidence with user preference characteristics.

Higher scores indicate products that better match both the recommendation model and the user's historical behavior.

In [ ]:
#giving weights
recommendations_df["personalization_score"] = (
    0.60 * recommendations_df["updated_score"]    #updated_score comes from the Feedback System
    + 0.25 * recommendations_df["avg_feedback_score"] #how positively the user has interacted overall.
    + 0.15 * recommendations_df["avg_updated_score"] #user's average recommendation quality across all recommendations
)

recommendations_df["personalization_score"] = (
    recommendations_df["personalization_score"]
    .clip(0, 1)
)

recommendations_df.head()

In [ ]:
recommendations_df = recommendations_df.sort_values(
    ["visitorid", "personalization_score"],
    ascending=[True, False]
)

recommendations_df["personalized_rank"] = (
    recommendations_df
    .groupby("visitorid")  #personalized items per user 
    .cumcount()   #count recommended products for particular user 
    + 1 #starts from 1 not 0 
)

recommendations_df.head(15)

## Extract Top-10 Personalized Recommendations

In production systems, only the highest-ranked products are presented to users.

Therefore, we retain the top ten personalized recommendations for every user.

In [ ]:
top10_recommendations = (
    recommendations_df[
        recommendations_df["personalized_rank"] <= 10
    ]
)

top10_recommendations.head(20)

## Analyze Personalized Recommendations

The distribution of personalization scores provides insight into the confidence of the final recommendation list.

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    recommendations_df["personalization_score"],
    bins=20
)

plt.title("Distribution of Personalization Scores")

plt.xlabel("Personalization Score")

plt.ylabel("Frequency")

plt.show()

## Save User Preference Profiles

The generated user profiles are saved for future analysis and deployment.

In [ ]:
user_profiles.to_csv(
    "../data/features/user_profiles.csv",
    index=False
)

print("User profiles saved successfully.")

## Save Final Personalized Recommendations

The final recommendation list contains products ranked according to both recommendation confidence and user preferences.

This dataset will be used by the API and deployment components of the project.

In [ ]:
top10_recommendations.to_csv(
    "../data/features/personalized_recommendations_final.csv",
    index=False
)

print("Final personalized recommendations saved successfully.")

### user_profiles.csv: One row per user containing aggregated preference statistics.
### personalized_recommendations_final.csv: Top-ranked personalized